In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [ ]:
# uncomment to use hosted db
os.environ["BIRDDOG_NOCODB_ENV"] = "CLOUD_PROD"
print(f"using {os.environ.get('BIRDDOG_NOCODB_ENV', 'LOCAL')} nocodb")


In [3]:
from birddog.database import Database
from birddog.runtime import Runtime
from birddog.database_updater import DatabaseUpdater
from birddog.wiki import (
    get_root_label,
    page_label,
    sequential_page_label,
    )

2026-07-11 08:37:04,519 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-07-11 08:37:04,531 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-07-11 08:37:04,726 [INFO] Translation is enabled. Using GCP translator
2026-07-11 08:37:04,727 [INFO] Using Google Cloud translation API
2026-07-11 08:37:04,727 [INFO] GoogleCloudTranslator using REST API


In [9]:
db = Database()

2026-07-11 09:35:19,245 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-07-11 09:35:19,422 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   49.60     0.45    39.00       0.00           24


In [ ]:
def snapshot_training_set(db):
    cursor = None
    while True:
        recs, cursor = db.scan(
            "Documents",
            view_name="Training Candidates",
            limit=500,
            cursor=cursor,
            fields=[
                "url",
                "sha1_hash",
                "title",
                "page_description",
                "page_native_description",
                "years",
                "root_label",
                "label",
                "seq_label",
                "level",
                "doc_type",
                "content_code",
                "process_code",
            ],
        )
        if not recs:
            break
        print(f"copying {len(recs)} records")
        db.write("ML Document Set", recs)
        if not cursor:
            break

In [ ]:
#snapshot_training_set(db)

In [ ]:
import hashlib

TEST_ARCHIVES = {"DAZHO", "DAZPO", "CDIAK"}  # ~11.5K records, ~7% of 164,805

def assign_split(root_label, label):
    if not root_label or not label:
        return "exclude"  # can't be fond-grouped; likely cleaned up in next rebuild
    if root_label in TEST_ARCHIVES:
        return "test"
    parts = label.split("/")
    fond_key = "/".join(parts[:2]) if len(parts) >= 2 else root_label
    h = int(hashlib.md5(fond_key.encode()).hexdigest(), 16)
    return "validation" if h % 100 < 15 else "train"

In [5]:
doc_ids = db.get_all_ids("ML Document Set")

2026-07-11 08:38:05,447 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   24.00     1.90    39.00       0.00           24


In [ ]:
recs = db.read(
    "ML Document Set",
    doc_ids,
    fields=[
            "url",
            "root_label",
            "label",
            "doc_type",
            "content_code",
            "process_code",
        ],
    )

In [ ]:
recs[:3]

In [ ]:
def get_fond_label(label):
    return "/".join(label.split("/")[:2])

In [ ]:
for rec in recs:
    rec["fond_label"] = get_fond_label(rec.get("label") or "")

In [ ]:
recs[:10]

In [ ]:
fond_labels = { rec["fond_label"] for rec in out_recs }

In [ ]:
len(fond_labels)

In [ ]:
def normalize_dist(dist, count):
    return { k: float(v)/float(count) for k, v in dist.items() }
    
def get_margin_dist(recs, normalize=False):
    margin_dist = {}
    for rec in recs:
        for field in ["doc_type", "content_code", "process_code"]:
            value = rec.get(field)
            if isinstance(value, list):
                for item in value:
                    margin_dist[item] = (margin_dist.get(item) or 0) + 1
            elif value:
                margin_dist[value] = (margin_dist.get(value) or 0) + 1
    if normalize:
        margin_dist = normalize_dist(margin_dist, len(recs))
    return margin_dist, len(recs)

In [ ]:
global_md, global_total = get_margin_dist(recs)
global_md, global_total

In [ ]:
root_labels = list({ r.get("root_label") for r in recs if r.get("root_label") })

In [ ]:
root_md = {}
root_count = {}
for label in root_labels:
    dist, count = get_margin_dist([rec for rec in recs if rec.get("root_label") == label])
    root_md[label] = dist
    root_count[label] = count

In [ ]:
def is_assigned_to(group_label, ml_split, assignment):
    return ml_split.get(group_label) == assignment

def get_split_dists(recs, ml_split, label_field="root_label"):
    train_dist, _ = get_margin_dist([rec for rec in recs if is_assigned_to(rec.get(label_field), ml_split, "train")], normalize=True)
    test_dist, _ = get_margin_dist([rec for rec in recs if is_assigned_to(rec.get(label_field), ml_split, "test")], normalize=True)
    return train_dist, test_dist

def split_score(recs, ml_split, label_field="root_label"):
    train_dist, test_dist = get_split_dists(recs, ml_split, label_field=label_field)
    return sum([
        abs(train_dist.get(v, 0.) - test_dist.get(v, 0.))
        for v in ["V", "C", "L", "O"]
    ])

In [ ]:
import random
def choose_split(recs, label_field="root_label", train_target=.9):
    label_list = list({ rec[label_field] for rec in recs if rec.get(label_field)})
    random.shuffle(label_list)

    label_count = {}
    for label in label_list:
        _, count = get_margin_dist([rec for rec in recs if rec.get(label_field) == label])
        label_count[label] = count
        
    ml_split = { }
    test_target = 1. - train_target
    count = 0
    train_count = 0
    test_count = 0
    for label in label_list:
        lc = label_count[label]
        count += lc
        train_fraction = float(train_count + lc) / float(count) 
        test_fraction = float(test_count + lc) / float(count) 
        if abs(train_fraction - train_target) < abs(test_fraction - test_target):
            train_count += lc
            ml_split[label] = "train"
        else:
            test_count += lc
            ml_split[label] = "test"
    print(f"totals train={train_count}, test={test_count}, train_frac={float(train_count)/float(count)}, test_frac={float(test_count)/float(count)}")
    return ml_split

In [ ]:
def search_for_split(recs, label_field="root_label"):
    best_score = 100.
    best_split = None
    for i in range(50):
        ml_split = choose_split(recs, label_field=label_field)
        score = split_score(recs, ml_split, label_field=label_field)
        print(f"split score: {score}")
        if score < best_score:
            best_score = score
            best_split = ml_split
    return best_split, best_score

In [ ]:
rsplit, rscore = search_for_split(recs, label_field="root_label")

In [ ]:
train_test_split = rsplit

In [ ]:
train_val_recs = [rec for rec in recs if is_assigned_to(rec.get("root_label"), train_test_split, "train")]

In [ ]:
test_recs = [rec for rec in recs if is_assigned_to(rec.get("root_label"), train_test_split, "test")]

In [ ]:
fsplit, fscore = search_for_split(train_val_recs, label_field="fond_label")

In [ ]:
fscore

In [ ]:
val_recs = [rec for rec in train_val_recs if is_assigned_to(rec.get("fond_label"), fsplit, "test")]

In [ ]:
train_recs = [rec for rec in train_val_recs if is_assigned_to(rec.get("fond_label"), fsplit, "train")]

In [ ]:
get_margin_dist(train_recs, normalize=True)

In [ ]:
get_margin_dist(val_recs, normalize=True)

In [ ]:
global_dist, _ = get_margin_dist(recs, normalize=True)
global_dist

In [ ]:
out_recs = []
out_recs.extend([
    { "Id": r["Id"], "url": r["url"], "ml_split": "train" } for r in train_recs 
    ])
out_recs.extend([
    { "Id": r["Id"], "url": r["url"], "ml_split": "test" } for r in test_recs 
    ])
out_recs.extend([
    { "Id": r["Id"], "url": r["url"], "ml_split": "validation" } for r in val_recs 
    ])
len(out_recs)

In [ ]:
out_recs_by_id =  {r["Id"] for r in out_recs}
exclude_recs = [ r for r in recs if r["Id"] not in out_recs_by_id ]
len(exclude_recs) + len(out_recs), len(recs)

In [ ]:
out_recs.extend([
    { "Id": r["Id"], "url": r["url"], "ml_split": "exclude" } for r in exclude_recs 
    ])
len(out_recs)

In [ ]:
out_recs[:5]

In [ ]:
cursor = 0
batch_size = 1000

In [ ]:
while True:
    batch = out_recs[cursor:(cursor+batch_size)]
    if not batch:
        break
    print(cursor)
    db.write("ML Document Set", batch)
    cursor += batch_size

In [ ]:
len(train_recs)

In [ ]:
len(test_recs)

In [ ]:
len(val_recs)

In [ ]:
len(exclude_recs)

In [6]:
recs = db.read(
    "ML Document Set",
    doc_ids,
    fields=[
            "url",
        ],
    )

2026-07-11 08:39:05,482 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   28.00     6.13    38.45       0.00           24


In [7]:
src_doc_ids = db.lookup("Documents", [r["url"] for r in recs])

2026-07-11 08:41:06,198 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   34.00    11.07    39.00       0.00           24
2026-07-11 08:42:06,229 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   39.00    26.22    38.60       0.00           24
2026-07-11 08:43:06,292 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   45.00    27.67    38.19       0.00           24
2026-07-11 08:44:06,296 [

In [8]:
len(src_doc_ids)

164805

In [10]:
src_recs = db.read("Documents", src_doc_ids, fields=["page_count", "comments"])

2026-07-11 09:36:19,460 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   53.60    11.29    39.00       0.00           24


In [11]:
src_recs[:3]

[{'Id': 8295, 'comments': None, 'page_count': 491},
 {'Id': 13739, 'comments': None, 'page_count': 489},
 {'Id': 13941, 'comments': '[AL] Has Jewish names', 'page_count': 414}]

In [12]:
recs[:3]

[{'Id': 1,
  'url': 'https://commons.wikimedia.org/wiki/File:ДАЗпО_Р-5593-24-53_Книга_реєстрації_актів_цивільного_стану_про_народження_(1936).pdf'},
 {'Id': 101,
  'url': 'https://uk.wikisource.org/wiki/File:ДАДнО_Р-6508-2-149_Книга_державної_реєстрації_актів_цивільного_стану_про_народження_Красногвардійського..._(1930).pdf'},
 {'Id': 201,
  'url': 'https://uk.wikisource.org/wiki/File:ДАДнО_Р-6508-7-601_Книга_реєстрації_актів_про_народження_Кіровський_район_(1937).pdf'}]

In [13]:
src_recs_by_id = { r["Id"]: r for r in src_recs }

In [14]:
out_recs = [{
    "url": rec["url"],
    "page_count": src_rec["page_count"],
    "comments": src_rec["comments"],
    } for rec, src_rec in zip(recs, src_recs)]

In [15]:
out_recs[:3]

[{'url': 'https://commons.wikimedia.org/wiki/File:ДАЗпО_Р-5593-24-53_Книга_реєстрації_актів_цивільного_стану_про_народження_(1936).pdf',
  'page_count': 491,
  'comments': None},
 {'url': 'https://uk.wikisource.org/wiki/File:ДАДнО_Р-6508-2-149_Книга_державної_реєстрації_актів_цивільного_стану_про_народження_Красногвардійського..._(1930).pdf',
  'page_count': 489,
  'comments': None},
 {'url': 'https://uk.wikisource.org/wiki/File:ДАДнО_Р-6508-7-601_Книга_реєстрації_актів_про_народження_Кіровський_район_(1937).pdf',
  'page_count': 414,
  'comments': '[AL] Has Jewish names'}]

In [17]:
#cursor = 0
#batch = 1000
#while cursor < len(out_recs):
#    print(cursor)
#    db.write("ML Document Set", out_recs[cursor:(cursor+batch)])
#    cursor += batch

In [18]:
urls = [r["url"] for r in recs]

In [19]:
import random
random.shuffle(urls)

In [20]:
N = 20
d1 = db.lookup("Documents", urls[:N])
d2 = db.lookup("ML Document Set", urls[:N])

2026-07-11 11:06:17,361 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                  100.00     0.30    37.36       0.00           24


In [21]:
r1 = db.read("Documents", d1, fields=["page_count", "comments"])

2026-07-11 11:08:18,989 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                  100.00     0.05    39.00       0.00           24


In [22]:
r2 = db.read("ML Document Set", d2, fields=["page_count", "comments"])

In [23]:
for rec1, rec2 in zip(r1, r2):
    if rec1["page_count"] != rec2["page_count"]:
        print("error", rec1, rec2)
    if rec1["comments"] != rec2["comments"]:
        print("error", rec1, rec2)